# Panel Comparison with Time Shift

This notebook loads panel 1 and panel 2 data and creates overlay visualizations for each channel to compare the time series and measure time delays between panels.

Adjust the `TIME_SHIFT_SECONDS` variable below to shift panel 2's data and find the optimal alignment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import re
import yaml

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configuration
DATA_DIR = Path('..') / Path('data')
PROCESSED_DIR = DATA_DIR / 'processed'
CONFIG_PATH = Path('..') / 'config' / 'default.yaml'

# Load timeframe from config (default.yaml known_anomalies first entry)
with open(CONFIG_PATH) as f:
    _config = yaml.safe_load(f)
_first = _config.get("known_anomalies", [{}])[0]
CONFIG_PLOT_START = _first.get("start", "2025-08-05T00:00:00+00:00")
CONFIG_PLOT_END = _first.get("end", "2025-08-05T23:59:59+00:00")

# ===== CONFIGURATION: Adjust time shift to align panels =====
# Positive values shift panel 2 forward in time (later)
# Negative values shift panel 2 backward in time (earlier)
TIME_SHIFT_SECONDS = 0  # Modify this value to shift panel 2 (e.g., 300 for 5 minutes)

# Timeframe: True = use config (default.yaml), False = use PLOT_DAY_OFFSET (current notebook behavior)
USE_CONFIG_TIMEFRAME = True

# Select which day to plot when USE_CONFIG_TIMEFRAME is False (0 = first day of overlap, 1 = second day, etc.)
PLOT_DAY_OFFSET = 10  # Modify this to plot a different day
# ============================================================

# File paths for processed data
PANEL_1_PROCESSED = PROCESSED_DIR / "panel_1.parquet"
PANEL_2_PROCESSED = PROCESSED_DIR / "panel_2.parquet"

print(f"Time shift: {TIME_SHIFT_SECONDS} seconds ({TIME_SHIFT_SECONDS/60:.1f} minutes)")
print(f"Panel 1 data: {PANEL_1_PROCESSED}")
print(f"Panel 2 data: {PANEL_2_PROCESSED}")

In [ ]:
# Load processed data
print("Loading processed data...")
df_panel_1 = pd.read_parquet(PANEL_1_PROCESSED)
df_panel_2 = pd.read_parquet(PANEL_2_PROCESSED)

# Ensure timestamp index is properly set
if not isinstance(df_panel_1.index, pd.DatetimeIndex):
    if df_panel_1.index.name == 'timestamp':
        df_panel_1.index = pd.to_datetime(df_panel_1.index)
    else:
        raise ValueError("Panel 1 index is not a DatetimeIndex and has no 'timestamp' name")

if not isinstance(df_panel_2.index, pd.DatetimeIndex):
    if df_panel_2.index.name == 'timestamp':
        df_panel_2.index = pd.to_datetime(df_panel_2.index)
    else:
        raise ValueError("Panel 2 index is not a DatetimeIndex and has no 'timestamp' name")

print(f"\nPanel 1:")
print(f"  Rows: {len(df_panel_1)}")
print(f"  Date range: {df_panel_1.index.min()} to {df_panel_1.index.max()}")
print(f"  Columns: {len(df_panel_1.columns)}")
print(f"  Column names: {list(df_panel_1.columns)[:5]}..." if len(df_panel_1.columns) > 5 else f"  Column names: {list(df_panel_1.columns)}")

print(f"\nPanel 2:")
print(f"  Rows: {len(df_panel_2)}")
print(f"  Date range: {df_panel_2.index.min()} to {df_panel_2.index.max()}")
print(f"  Columns: {len(df_panel_2.columns)}")
print(f"  Column names: {list(df_panel_2.columns)[:5]}..." if len(df_panel_2.columns) > 5 else f"  Column names: {list(df_panel_2.columns)}")

In [ ]:
# Group columns by channel ID
# Column pattern: channel_{channel_id}_{key}

def extract_channel_info(column_name):
    """Extract channel ID and measurement key from column name."""
    match = re.match(r'channel_(\d+)_(.+)', column_name)
    if match:
        return int(match.group(1)), match.group(2)
    return None, None

# Group columns by channel for both panels
channels_p1 = {}
channels_p2 = {}

for col in df_panel_1.columns:
    channel_id, key = extract_channel_info(col)
    if channel_id is not None:
        if channel_id not in channels_p1:
            channels_p1[channel_id] = []
        channels_p1[channel_id].append((col, key))

for col in df_panel_2.columns:
    channel_id, key = extract_channel_info(col)
    if channel_id is not None:
        if channel_id not in channels_p2:
            channels_p2[channel_id] = []
        channels_p2[channel_id].append((col, key))

# Get all unique channel IDs (union of both panels)
all_channels = sorted(set(list(channels_p1.keys()) + list(channels_p2.keys())))

print(f"Found {len(all_channels)} channels: {all_channels}")
print("\nChannel breakdown:")
for channel_id in all_channels:
    p1_keys = [key for _, key in channels_p1.get(channel_id, [])]
    p2_keys = [key for _, key in channels_p2.get(channel_id, [])]
    print(f"  Channel {channel_id}:")
    print(f"    Panel 1: {p1_keys}")
    print(f"    Panel 2: {p2_keys}")

In [ ]:
# Apply time shift to panel 2
df_panel_2_shifted = df_panel_2.copy()
df_panel_2_shifted.index = df_panel_2_shifted.index + pd.Timedelta(seconds=TIME_SHIFT_SECONDS)

print(f"Applied time shift of {TIME_SHIFT_SECONDS} seconds to Panel 2")
print(f"\nPanel 2 (shifted) date range: {df_panel_2_shifted.index.min()} to {df_panel_2_shifted.index.max()}")

# Calculate overlap period
overlap_start = max(df_panel_1.index.min(), df_panel_2_shifted.index.min())
overlap_end = min(df_panel_1.index.max(), df_panel_2_shifted.index.max())

print(f"\nOverlap period: {overlap_start} to {overlap_end}")
print(f"Overlap duration: {overlap_end - overlap_start}")

In [ ]:
# Create overlay plots for each channel
# Timeframe: from config (default.yaml) or from overlap + PLOT_DAY_OFFSET
if USE_CONFIG_TIMEFRAME:
    plot_day_start = pd.Timestamp(CONFIG_PLOT_START)
    plot_day_end = pd.Timestamp(CONFIG_PLOT_END)
else:
    overlap_start = max(df_panel_1.index.min(), df_panel_2_shifted.index.min())
    overlap_end = min(df_panel_1.index.max(), df_panel_2_shifted.index.max())
    plot_day_start = overlap_start + pd.Timedelta(days=PLOT_DAY_OFFSET)
    plot_day_end = plot_day_start + pd.Timedelta(days=1)
    plot_day_start = max(plot_day_start, overlap_start)
    plot_day_end = min(plot_day_end, overlap_end)

# Filter data to selected day
df_p1_overlap = df_panel_1.loc[plot_day_start:plot_day_end]
df_p2_overlap = df_panel_2_shifted.loc[plot_day_start:plot_day_end]

if USE_CONFIG_TIMEFRAME:
    print(f"Plotting data for config timeframe: {plot_day_start.date()}")
else:
    print(f"Plotting data for: {plot_day_start.date()} (day {PLOT_DAY_OFFSET} of overlap period)")
print(f"Time range: {plot_day_start} to {plot_day_end}")

# Determine grid layout (3 rows x 2 cols for 5 channels, with one empty)
n_channels = len(all_channels)
n_cols = 2
n_rows = (n_channels + 1) // 2  # Round up

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
if n_channels == 1:
    axes = [axes]
else:
    axes = axes.flatten()

# Plot each channel
for idx, channel_id in enumerate(all_channels):
    ax = axes[idx]
    
    # Get columns for this channel
    p1_cols = [col for col, _ in channels_p1.get(channel_id, [])]
    p2_cols = [col for col, _ in channels_p2.get(channel_id, [])]
    
    # Plot panel 1 data
    for col in p1_cols:
        if col in df_p1_overlap.columns:
            ax.plot(df_p1_overlap.index, df_p1_overlap[col], 
                   label=f"Panel 1: {col.split('_', 2)[-1]}", 
                   alpha=0.7, linewidth=1.5)
    
    # Plot panel 2 (shifted) data
    for col in p2_cols:
        if col in df_p2_overlap.columns:
            ax.plot(df_p2_overlap.index, df_p2_overlap[col], 
                   label=f"Panel 2: {col.split('_', 2)[-1]}", 
                   alpha=0.7, linewidth=1.5, linestyle='--')
    
    ax.set_title(f'Channel {channel_id} (Shift: {TIME_SHIFT_SECONDS}s)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Format x-axis dates
    ax.tick_params(axis='x', rotation=45)
    
    # Set x-axis to show hours of the day
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))

# Hide unused subplots
for idx in range(n_channels, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\nPlotted {n_channels} channels with time shift of {TIME_SHIFT_SECONDS} seconds")

## Average difference over 24 hours by time shift

Sweep time shifts (same range as the interactive slider) and compute the mean absolute difference between panel 1 and panel 2 (shifted) over the config 24h window. One plot per channel; the minimum indicates the best-aligning shift for that channel.

In [ ]:
# Sweep params (same as interactive slider)
shift_min = -3600 * 24 * 4   # -4 days (seconds)
shift_max = 3600 * 24 * 4    # +4 days (seconds)
shift_step = 300             # 5 min steps

shift_seconds = np.arange(shift_min, shift_max + 1, shift_step)
plot_start = pd.Timestamp(CONFIG_PLOT_START)
plot_end = pd.Timestamp(CONFIG_PLOT_END)

# Slice both panels to the 24h window (panel 1 index is the reference)
df_p1_24h = df_panel_1.loc[plot_start:plot_end]
ref_index = df_p1_24h.index

# For each channel: [shift_seconds] -> average absolute difference
channel_diffs = {ch: np.full(len(shift_seconds), np.nan) for ch in all_channels}

for i, shift_sec in enumerate(shift_seconds):
    df_p2_shifted = df_panel_2.copy()
    df_p2_shifted.index = df_panel_2.index + pd.Timedelta(seconds=shift_sec)
    df_p2_24h = df_p2_shifted.loc[plot_start:plot_end].sort_index()
    # Align panel 2 to panel 1 timestamps (nearest) via merge_asof
    left = pd.DataFrame({"timestamp": ref_index}).sort_values("timestamp")
    right = df_p2_24h.reset_index().sort_values("timestamp")
    merged = pd.merge_asof(left, right, on="timestamp", direction="nearest")
    df_p2_aligned = merged.set_index("timestamp")[df_p2_24h.columns].reindex(ref_index)

    for ch in all_channels:
        p1_cols = [c for c, _ in channels_p1.get(ch, [])]
        p2_cols = [c for c, _ in channels_p2.get(ch, [])]
        if not p1_cols or not p2_cols:
            continue
        # Pair by column name (same names in both panels)
        diffs = []
        for c in p1_cols:
            if c not in df_p2_aligned.columns:
                continue
            d = (df_p1_24h[c] - df_p2_aligned[c]).abs()
            diffs.append(d.mean())
        if diffs:
            channel_diffs[ch][i] = np.nanmean(diffs)

# Plot: one subplot per channel
n_channels = len(all_channels)
n_cols = 2
n_rows = (n_channels + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
if n_channels == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for idx, ch in enumerate(all_channels):
    ax = axes[idx]
    ax.plot(shift_seconds / 3600, channel_diffs[ch], linewidth=1)
    best_idx = np.nanargmin(channel_diffs[ch])
    best_shift = shift_seconds[best_idx]
    best_val = channel_diffs[ch][best_idx]
    ax.axvline(best_shift / 3600, color="red", linestyle="--", alpha=0.7, label=f"Best: {best_shift/3600:.2f} h")
    ax.set_title(f"Channel {ch} — avg |P1−P2| over 24h")
    ax.set_xlabel("Time shift (hours)")
    ax.set_ylabel("Mean absolute difference")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)

for j in range(n_channels, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

# Summary: best shift per channel
print("Best time shift (hours) per channel (min avg |P1−P2| over 24h):")
for ch in all_channels:
    best_idx = np.nanargmin(channel_diffs[ch])
    print(f"  Channel {ch}: {shift_seconds[best_idx]/3600:.2f} h ({shift_seconds[best_idx]} s)")

## Interactive Time Shift Adjustment (Optional)

Use the widget below to interactively adjust the time shift and see the alignment in real-time.

In [ ]:
# Interactive widget for time shift adjustment
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    
    def plot_with_shift(shift_seconds):
        """Replot with the given time shift."""
        # Apply shift
        df_p2_shifted = df_panel_2.copy()
        df_p2_shifted.index = df_panel_2.index + pd.Timedelta(seconds=shift_seconds)
        
        # Timeframe: from config or overlap + PLOT_DAY_OFFSET
        if USE_CONFIG_TIMEFRAME:
            plot_day_start = pd.Timestamp(CONFIG_PLOT_START)
            plot_day_end = pd.Timestamp(CONFIG_PLOT_END)
        else:
            overlap_start = max(df_panel_1.index.min(), df_p2_shifted.index.min())
            overlap_end = min(df_panel_1.index.max(), df_p2_shifted.index.max())
            plot_day_start = overlap_start + pd.Timedelta(days=PLOT_DAY_OFFSET)
            plot_day_end = plot_day_start + pd.Timedelta(days=1)
            plot_day_start = max(plot_day_start, overlap_start)
            plot_day_end = min(plot_day_end, overlap_end)
        
        df_p1_overlap = df_panel_1.loc[plot_day_start:plot_day_end]
        df_p2_overlap = df_p2_shifted.loc[plot_day_start:plot_day_end]
        
        # Create plots
        n_channels = len(all_channels)
        n_cols = 2
        n_rows = (n_channels + 1) // 2
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
        if n_channels == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        for idx, channel_id in enumerate(all_channels):
            ax = axes[idx]
            
            p1_cols = [col for col, _ in channels_p1.get(channel_id, [])]
            p2_cols = [col for col, _ in channels_p2.get(channel_id, [])]
            
            for col in p1_cols:
                if col in df_p1_overlap.columns:
                    ax.plot(df_p1_overlap.index, df_p1_overlap[col], 
                           label=f"Panel 1: {col.split('_', 2)[-1]}", 
                           alpha=0.7, linewidth=1.5)
            
            for col in p2_cols:
                if col in df_p2_overlap.columns:
                    ax.plot(df_p2_overlap.index, df_p2_overlap[col], 
                           label=f"Panel 2: {col.split('_', 2)[-1]}", 
                           alpha=0.7, linewidth=1.5, linestyle='--')
            
            ax.set_title(f'Channel {channel_id} (Shift: {shift_seconds}s)', fontsize=12, fontweight='bold')
            ax.set_xlabel('Time')
            ax.set_ylabel('Value')
            ax.legend(loc='best', fontsize=8)
            ax.grid(True, alpha=0.3)
            ax.tick_params(axis='x', rotation=45)
            
            # Set x-axis to show hours of the day
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        
        for idx in range(n_channels, len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        plt.show()
    
    # Create slider widget
    # Estimate reasonable range: ±1 hour in 30-second steps
    shift_slider = widgets.IntSlider(
        value=TIME_SHIFT_SECONDS,
        min=-3600*24*4,  # -1 hour
        max=3600*24*4,   # +1 hour
        step=300,    # 5 min steps
        description='Time Shift (s):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    
    output = widgets.Output()
    
    def on_value_change(change):
        with output:
            clear_output(wait=True)
            plot_with_shift(change['new'])
    
    shift_slider.observe(on_value_change, names='value')
    
    display(shift_slider, output)
    # Initial plot
    plot_with_shift(TIME_SHIFT_SECONDS)
    
except ImportError:
    print("ipywidgets not available. Install with: pip install ipywidgets")
    print("Or use the TIME_SHIFT_SECONDS variable in the configuration cell above.")